# Agent Tools & Capabilities

Welcome to the second notebook in the Azure AI Agent Service tutorial! This notebook explores the powerful built-in tools that make agents truly capable.

## Prerequisites

Before starting this tutorial, make sure you've completed the **Agent Service Basics** notebook. You should be familiar with:
- Creating agents and threads
- Running conversations
- Basic function calling

## What You'll Learn

- **File Search**: Upload files and let agents search through them semantically
- **Code Interpreter**: Let agents write and execute Python code
- **Bing Grounding**: Connect agents to real-time web search
- **Azure AI Search**: Enterprise-grade search integration
- **Streaming Responses**: Real-time response generation

## Resources
- [File Search Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/file-search)
- [Code Interpreter Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/code-interpreter)
- [Bing Grounding Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/bing-grounding)

Let's explore these powerful capabilities!

## Setup and Imports

In [ ]:
import os
import json
from dotenv import load_dotenv

# Azure AI Projects SDK
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import (
    MessageRole,
    RunStatus,
    FileSearchTool,
    CodeInterpreterTool,
    BingGroundingTool,
    AzureAISearchTool,
    ToolSet,
    FilePurpose,
    VectorStoreDataSource,
    VectorStoreDataSourceAssetType
)
from azure.identity import DefaultAzureCredential

# Load environment variables
load_dotenv()

print("✅ Imports loaded successfully!")

In [ ]:
# Initialize the project client
project_client = AIProjectClient(
    credential=DefaultAzureCredential(),
    endpoint=os.environ["PROJECT_ENDPOINT"]
)

MODEL_DEPLOYMENT = os.getenv("MODEL_DEPLOYMENT_NAME", "gpt-4o")

print(f"✅ Connected to Azure AI Foundry")
print(f"📦 Using model: {MODEL_DEPLOYMENT}")

## 1. File Search with Vector Stores

The File Search tool allows agents to search through uploaded documents using semantic search. This is perfect for:
- Knowledge bases
- Documentation assistants
- Research helpers

### How It Works:
1. Upload files to Azure
2. Create a vector store from the files
3. Attach the vector store to an agent
4. The agent can now search through the documents

In [ ]:
# First, let's create a sample document to upload
sample_content = """# Azure AI Services Overview

## Introduction
Azure AI Services is a comprehensive suite of artificial intelligence tools and services 
offered by Microsoft Azure. It enables developers to build intelligent applications 
without requiring deep expertise in machine learning.

## Key Services

### Azure OpenAI Service
Access to GPT-4, GPT-4o, and other powerful language models with enterprise-grade security.
- Chat completions
- Embeddings
- Fine-tuning capabilities

### Azure AI Search
Enterprise-grade search powered by AI:
- Full-text search
- Vector search (semantic)
- Hybrid search combining both

### Azure AI Agent Service
Build and deploy AI agents with:
- Persistent conversation threads
- Built-in tools (File Search, Code Interpreter)
- Multi-agent orchestration

### Azure AI Document Intelligence
Extract information from documents:
- Invoice processing
- Receipt scanning
- Custom document models

## Pricing
Azure AI Services uses a pay-as-you-go model. Key factors:
- Token usage for language models
- Number of API calls
- Storage for uploaded files
- Vector store operations

## Best Practices
1. Use managed identity for authentication when possible
2. Implement rate limiting for production applications
3. Monitor usage with Azure Monitor
4. Use content safety features to filter harmful content
5. Leverage caching to reduce costs

## Getting Started
1. Create an Azure AI Foundry project
2. Deploy a language model
3. Use the Python SDK to build your application
4. Test locally before deploying to production
"""

# Save to a temporary file
with open("resources/azure_ai_overview.md", "w") as f:
    f.write(sample_content)

print("✅ Sample document created!")

In [ ]:
# Upload the file to Azure
file = project_client.agents.upload_file_and_poll(
    file_path="resources/azure_ai_overview.md",
    purpose=FilePurpose.AGENTS
)

print(f"✅ File uploaded!")
print(f"   ID: {file.id}")
print(f"   Name: {file.filename}")
print(f"   Size: {file.bytes} bytes")

In [ ]:
# Create a vector store from the uploaded file
# This indexes the content for semantic search
vector_store = project_client.agents.create_vector_store_and_poll(
    file_ids=[file.id],
    name="AzureDocsKnowledgeBase"
)

print(f"✅ Vector store created!")
print(f"   ID: {vector_store.id}")
print(f"   Name: {vector_store.name}")
print(f"   Files: {vector_store.file_counts}")

In [ ]:
# Create an agent with File Search capability
file_search_tool = FileSearchTool(vector_store_ids=[vector_store.id])

file_search_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="DocumentExpert",
    instructions="""You are a helpful documentation assistant with access to Azure AI documentation.
    
When answering questions:
1. Always search the knowledge base first
2. Quote relevant sections when helpful
3. Be accurate - only answer based on what's in the documents
4. If information isn't in the documents, say so clearly""",
    tools=[file_search_tool],
    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}}
)

print(f"✅ File Search agent created!")
print(f"   ID: {file_search_agent.id}")

In [ ]:
# Helper function for file search agent
def ask_documents(question: str, thread_id: str, agent_id: str) -> str:
    """Ask a question about the uploaded documents."""
    project_client.agents.create_message(
        thread_id=thread_id,
        role=MessageRole.USER,
        content=question
    )
    
    run = project_client.agents.create_and_process_run(
        thread_id=thread_id,
        agent_id=agent_id
    )
    
    messages = project_client.agents.list_messages(thread_id=thread_id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    # Check for annotations (citations)
    if response and response.text.annotations:
        print("📎 Citations:")
        for anno in response.text.annotations:
            if hasattr(anno, 'file_citation'):
                print(f"   - {anno.file_citation.file_id}")
    
    return response.text.value if response else "No response"

In [ ]:
# Test the file search agent
file_search_thread = project_client.agents.create_thread()

questions = [
    "What is Azure AI Agent Service and what are its main features?",
    "What are the best practices for using Azure AI Services?",
    "How does pricing work for Azure AI Services?"
]

for question in questions:
    print(f"❓ Question: {question}")
    response = ask_documents(question, file_search_thread.id, file_search_agent.id)
    print(f"📚 Answer: {response[:500]}..." if len(response) > 500 else f"📚 Answer: {response}")
    print("\n" + "="*60 + "\n")

## 2. Code Interpreter

The Code Interpreter tool allows agents to write and execute Python code. This is powerful for:
- Data analysis
- Mathematical calculations
- Generating charts and visualizations
- Processing files

In [ ]:
# Create an agent with Code Interpreter
code_interpreter_tool = CodeInterpreterTool()

code_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="DataAnalyst",
    instructions="""You are a data analysis expert with Python coding capabilities.
    
When users ask for analysis or calculations:
1. Write clean, well-commented Python code
2. Execute the code to get results
3. Explain the results clearly
4. Create visualizations when helpful
5. Handle errors gracefully and explain what went wrong

Available libraries: numpy, pandas, matplotlib, scipy""",
    tools=[code_interpreter_tool]
)

print(f"✅ Code Interpreter agent created!")
print(f"   ID: {code_agent.id}")

In [ ]:
# Helper function for code interpreter
def run_code_task(task: str, thread_id: str, agent_id: str):
    """Run a task that may involve code execution."""
    project_client.agents.create_message(
        thread_id=thread_id,
        role=MessageRole.USER,
        content=task
    )
    
    run = project_client.agents.create_and_process_run(
        thread_id=thread_id,
        agent_id=agent_id
    )
    
    # Get the response
    messages = project_client.agents.list_messages(thread_id=thread_id)
    response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
    
    # Check for generated files (like images)
    if response:
        for item in response.content:
            if hasattr(item, 'image_file'):
                print(f"📊 Generated image: {item.image_file.file_id}")
    
    return response.text.value if response else "No response"

In [ ]:
# Test the code interpreter agent
code_thread = project_client.agents.create_thread()

# Mathematical calculation
print("🔢 Task 1: Mathematical Calculation")
print("-" * 40)
task = "Calculate the compound interest on $10,000 invested at 7% annual rate for 10 years, compounded monthly."
print(f"📝 Task: {task}")
response = run_code_task(task, code_thread.id, code_agent.id)
print(f"🤖 Response:\n{response}")
print("\n" + "="*60 + "\n")

In [ ]:
# Statistical analysis
print("📊 Task 2: Statistical Analysis")
print("-" * 40)
task = """Generate a sample dataset of 100 normally distributed values with mean 50 and std 15.
Calculate the mean, median, mode, and standard deviation.
Then create a histogram visualization."""
print(f"📝 Task: {task}")
response = run_code_task(task, code_thread.id, code_agent.id)
print(f"🤖 Response:\n{response}")
print("\n" + "="*60 + "\n")

In [ ]:
# Data processing task
print("🔄 Task 3: Data Processing")
print("-" * 40)
task = """Create a simple sales dataset with the following columns: month (Jan-Dec), 
revenue (random values between 50000-150000), and expenses (60-80% of revenue).
Calculate the profit for each month and identify the most and least profitable months."""
print(f"📝 Task: {task}")
response = run_code_task(task, code_thread.id, code_agent.id)
print(f"🤖 Response:\n{response}")

## 3. Bing Grounding (Web Search)

The Bing Grounding tool allows agents to search the web for current information. This is essential for:
- Real-time information queries
- Fact-checking
- Current events
- Product/service information

**Note**: This requires a Bing Search resource connected to your Azure AI Foundry project.

In [ ]:
# Check if Bing connection is configured
BING_CONNECTION = os.getenv("BING_CONNECTION_NAME")

if BING_CONNECTION:
    # Create Bing Grounding tool
    bing_tool = BingGroundingTool(connection_id=BING_CONNECTION)
    
    bing_agent = project_client.agents.create_agent(
        model=MODEL_DEPLOYMENT,
        name="WebResearcher",
        instructions="""You are a web research assistant with access to Bing search.
        
    When answering questions:
    1. ALWAYS use the Bing search tool for factual queries
    2. Cite your sources with URLs when possible
    3. Indicate when information might be outdated
    4. Be clear about what's from search vs your knowledge
    5. For opinions or subjective questions, present multiple viewpoints""",
        tools=[bing_tool]
    )
    
    print(f"✅ Bing Grounding agent created!")
    print(f"   ID: {bing_agent.id}")
else:
    print("⚠️ BING_CONNECTION_NAME not configured")
    print("   To use Bing Grounding, add a Bing Search connection in Azure AI Foundry")
    print("   and set BING_CONNECTION_NAME in your .env file")

In [ ]:
# Test Bing Grounding (if available)
if BING_CONNECTION:
    bing_thread = project_client.agents.create_thread()
    
    def search_web(query: str) -> str:
        """Search the web using Bing Grounding."""
        project_client.agents.create_message(
            thread_id=bing_thread.id,
            role=MessageRole.USER,
            content=query
        )
        
        run = project_client.agents.create_and_process_run(
            thread_id=bing_thread.id,
            agent_id=bing_agent.id
        )
        
        messages = project_client.agents.list_messages(thread_id=bing_thread.id)
        response = messages.get_last_text_message_by_role(MessageRole.ASSISTANT)
        
        # Check for citations
        if response and response.text.annotations:
            print("🔗 Sources:")
            for anno in response.text.annotations:
                if hasattr(anno, 'url_citation'):
                    print(f"   - {anno.url_citation.url}")
        
        return response.text.value if response else "No response"
    
    # Test queries
    queries = [
        "What are the latest features in Azure AI Agent Service?",
        "What is the current weather in Seattle?"
    ]
    
    for query in queries:
        print(f"🔍 Query: {query}")
        response = search_web(query)
        print(f"🌐 Response: {response[:400]}..." if len(response) > 400 else f"🌐 Response: {response}")
        print("\n" + "="*60 + "\n")
else:
    print("⏭️ Skipping Bing Grounding test (not configured)")

## 4. Azure AI Search Integration

For enterprise scenarios, you can integrate Azure AI Search for:
- Large-scale document search
- Hybrid search (keyword + semantic)
- Custom indexes with facets and filters

**Note**: This requires an Azure AI Search resource connected to your project.

In [ ]:
# Check if Azure AI Search is configured
AI_SEARCH_CONNECTION = os.getenv("AZURE_AI_SEARCH_CONNECTION_NAME")
AI_SEARCH_INDEX = os.getenv("AZURE_AI_SEARCH_INDEX_NAME")

if AI_SEARCH_CONNECTION and AI_SEARCH_INDEX:
    # Create Azure AI Search tool
    ai_search_tool = AzureAISearchTool(
        connection_id=AI_SEARCH_CONNECTION,
        index_name=AI_SEARCH_INDEX
    )
    
    enterprise_search_agent = project_client.agents.create_agent(
        model=MODEL_DEPLOYMENT,
        name="EnterpriseSearchAssistant",
        instructions="""You are an enterprise search assistant with access to the company knowledge base.
        
    Guidelines:
    1. Search the index for relevant documents
    2. Synthesize information from multiple sources
    3. Always cite document sources
    4. Indicate confidence levels in your answers
    5. Suggest related topics for further research""",
        tools=[ai_search_tool]
    )
    
    print(f"✅ Azure AI Search agent created!")
    print(f"   ID: {enterprise_search_agent.id}")
    print(f"   Index: {AI_SEARCH_INDEX}")
else:
    print("⚠️ Azure AI Search not configured")
    print("   Set AZURE_AI_SEARCH_CONNECTION_NAME and AZURE_AI_SEARCH_INDEX_NAME in .env")

## 5. Streaming Responses

For a better user experience with long responses, you can stream the agent's output token by token.

In [ ]:
from azure.ai.projects.models import (
    MessageDeltaChunk,
    ThreadMessage,
    ThreadRun,
    RunStep
)

def stream_response(user_message: str, thread_id: str, agent_id: str):
    """Stream the agent's response in real-time."""
    # Add user message
    project_client.agents.create_message(
        thread_id=thread_id,
        role=MessageRole.USER,
        content=user_message
    )
    
    print("🤖 Assistant: ", end="", flush=True)
    
    # Stream the response
    with project_client.agents.create_stream(
        thread_id=thread_id,
        agent_id=agent_id
    ) as stream:
        for event_type, event_data in stream:
            if isinstance(event_data, MessageDeltaChunk):
                # Print each token as it arrives
                if event_data.delta.content:
                    for content in event_data.delta.content:
                        if hasattr(content, 'text') and content.text.value:
                            print(content.text.value, end="", flush=True)
    
    print()  # New line at the end

print("✅ Streaming function ready!")

In [ ]:
# Create a simple agent for streaming demo
streaming_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="StreamingAssistant",
    instructions="You are a helpful assistant. Provide detailed, well-structured responses."
)

streaming_thread = project_client.agents.create_thread()

print("✅ Streaming agent and thread created!")

In [ ]:
# Test streaming
print("📝 Testing streaming response...")
print("="*60)
print("👤 User: Explain the benefits of using cloud-hosted AI agents.")
print()

stream_response(
    "Explain the benefits of using cloud-hosted AI agents compared to self-hosted solutions.",
    streaming_thread.id,
    streaming_agent.id
)

## 6. Streaming with Tool Calls

You can also stream responses that include tool calls, allowing you to see both the tool execution and the response generation in real-time.

In [ ]:
from azure.ai.projects.models import (
    RunStepDeltaChunk,
    RunStepDeltaToolCallObject,
    RunStepToolCallDelta
)

def stream_with_events(user_message: str, thread_id: str, agent_id: str):
    """Stream response with detailed event handling."""
    project_client.agents.create_message(
        thread_id=thread_id,
        role=MessageRole.USER,
        content=user_message
    )
    
    with project_client.agents.create_stream(
        thread_id=thread_id,
        agent_id=agent_id
    ) as stream:
        for event_type, event_data in stream:
            # Handle different event types
            if isinstance(event_data, ThreadRun):
                print(f"📍 Run status: {event_data.status}")
            
            elif isinstance(event_data, RunStep):
                if event_data.type == "tool_calls":
                    print("🔧 Processing tool calls...")
            
            elif isinstance(event_data, RunStepDeltaChunk):
                if event_data.delta.step_details:
                    for tool_call in event_data.delta.step_details.tool_calls or []:
                        if hasattr(tool_call, 'code_interpreter'):
                            if tool_call.code_interpreter.input:
                                print(f"💻 Code: {tool_call.code_interpreter.input}")
            
            elif isinstance(event_data, MessageDeltaChunk):
                if event_data.delta.content:
                    for content in event_data.delta.content:
                        if hasattr(content, 'text') and content.text.value:
                            print(content.text.value, end="", flush=True)
    
    print()

print("✅ Event streaming function ready!")

In [ ]:
# Test streaming with code interpreter
print("📝 Testing streaming with Code Interpreter...")
print("="*60)

# Create a new thread for this test
code_stream_thread = project_client.agents.create_thread()

stream_with_events(
    "Calculate the first 10 numbers in the Fibonacci sequence and show them.",
    code_stream_thread.id,
    code_agent.id
)

## 7. Combining Multiple Tools

Agents can use multiple tools together for complex tasks.

In [ ]:
# Create an agent with multiple tools
multi_toolset = ToolSet()
multi_toolset.add(CodeInterpreterTool())
multi_toolset.add(FileSearchTool(vector_store_ids=[vector_store.id]))

multi_tool_agent = project_client.agents.create_agent(
    model=MODEL_DEPLOYMENT,
    name="ResearchAnalyst",
    instructions="""You are a research analyst with access to documentation and code execution.
    
Capabilities:
1. Search through documentation for information
2. Execute Python code for analysis and calculations
3. Combine information from multiple sources

Guidelines:
- Search documents first for context
- Use code for any calculations or data processing
- Provide comprehensive answers with sources""",
    toolset=multi_toolset,
    tool_resources={"file_search": {"vector_store_ids": [vector_store.id]}}
)

print(f"✅ Multi-tool agent created!")
print(f"   ID: {multi_tool_agent.id}")
print(f"   Tools: File Search, Code Interpreter")

In [ ]:
# Test multi-tool agent
multi_thread = project_client.agents.create_thread()

task = """Based on the Azure AI Services documentation, create a simple cost estimation:
1. First, search for pricing information in the docs
2. Then create a Python calculation for monthly costs assuming:
   - 1 million tokens per month
   - 1000 API calls per day
   - 10GB of file storage

Use reasonable estimates for pricing if exact numbers aren't available."""

print(f"📝 Complex Task: {task}")
print("="*60)

# Run with detailed event streaming
stream_with_events(task, multi_thread.id, multi_tool_agent.id)

## 8. Cleanup

In [ ]:
# Clean up all resources
print("🧹 Cleaning up resources...")

# Delete threads
threads = [
    file_search_thread.id,
    code_thread.id,
    streaming_thread.id,
    code_stream_thread.id,
    multi_thread.id
]

# Add optional threads if they exist
if BING_CONNECTION:
    threads.append(bing_thread.id)

for thread_id in threads:
    try:
        project_client.agents.delete_thread(thread_id)
        print(f"   ✅ Deleted thread: {thread_id[:20]}...")
    except Exception as e:
        print(f"   ⚠️ Could not delete thread: {e}")

# Delete agents
agents = [
    file_search_agent.id,
    code_agent.id,
    streaming_agent.id,
    multi_tool_agent.id
]

if BING_CONNECTION:
    agents.append(bing_agent.id)

for agent_id in agents:
    try:
        project_client.agents.delete_agent(agent_id)
        print(f"   ✅ Deleted agent: {agent_id[:20]}...")
    except Exception as e:
        print(f"   ⚠️ Could not delete agent: {e}")

# Delete vector store
try:
    project_client.agents.delete_vector_store(vector_store.id)
    print(f"   ✅ Deleted vector store: {vector_store.id[:20]}...")
except Exception as e:
    print(f"   ⚠️ Could not delete vector store: {e}")

# Delete uploaded file
try:
    project_client.agents.delete_file(file.id)
    print(f"   ✅ Deleted file: {file.id[:20]}...")
except Exception as e:
    print(f"   ⚠️ Could not delete file: {e}")

print("\n✅ Cleanup complete!")

## 🎉 Congratulations!

You've completed the Agent Tools & Capabilities tutorial! Here's what you've learned:

### ✅ Tools Covered:
1. **File Search** - Semantic search through uploaded documents with vector stores
2. **Code Interpreter** - Execute Python code for data analysis and calculations
3. **Bing Grounding** - Real-time web search for current information
4. **Azure AI Search** - Enterprise-grade search integration
5. **Streaming** - Real-time response generation

### 🔧 Key Concepts:
- Vector stores for document indexing
- Tool resources and configurations
- Event streaming for tool execution visibility
- Combining multiple tools for complex tasks

### 🚀 Next Steps:
Continue with the **Multi-Agent Orchestration** notebook to learn about:
- Connected agents (agents as tools)
- Sequential workflows
- Agent handoffs
- Human-in-the-loop patterns

### 📚 Additional Resources:
- [File Search Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/file-search)
- [Code Interpreter Documentation](https://learn.microsoft.com/en-us/azure/ai-foundry/agents/how-to/tools/code-interpreter)
- [Azure AI Foundry Samples](https://github.com/azure-ai-foundry/foundry-samples)

Great work! You're now ready to build powerful tool-enabled agents. 🚀